In [1]:
import json
import re
import pandas as pd

In [2]:
REQUESTS_PATH = "/Users/utkarshumang/my_projects/lead-enricher-ai-be/batch_api/jsonl/requests/no_data_batch_requests.jsonl"
RESPONSES_PATH = "/Users/utkarshumang/my_projects/lead-enricher-ai-be/batch_api/jsonl/results/cleaned/no_data_results_cleaned.jsonl"
CSV_PATH = "/Users/utkarshumang/my_projects/lead-enricher-ai-be/batch_api/csv/merged_apify_leads - apify_list_without_full_data.csv"
OUTPUT_CSV_PATH = "/Users/utkarshumang/my_projects/lead-enricher-ai-be/batch_api/csv/merged_apify_leads - apify_list_without_full_data(1).csv"

In [3]:
EXAMPLE_EMAIL = "marmaladeskies.store@gmail.com"
EMAIL_PATTERN = r"email\s*-\s*([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+)"

In [4]:
def resolve_lead_email(prompt: str) -> str | None:
    prompt_lower = prompt.lower()

    # Find ALL "email - xyz" occurrences
    matches = re.findall(EMAIL_PATTERN, prompt_lower)

    # Remove the example email explicitly
    candidates = [e for e in matches if e != EXAMPLE_EMAIL]

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        # Defensive: pick the last one (closest to lead data)
        return candidates[-1]

    return None

In [5]:
request_rows = []

with open(REQUESTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        prompt = obj["body"]["messages"][0]["content"]

        email = resolve_lead_email(prompt)

        if email:
            request_rows.append({
                "custom_id": obj["custom_id"],
                "email": email
            })

requests_df = pd.DataFrame(request_rows)

In [6]:
response_rows = []

with open(RESPONSES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("error") is None:
            content = obj["response"]["body"]["choices"][0]["message"]["content"]
            response_rows.append({
                "custom_id": obj["custom_id"],
                "final_name": content.strip()
            })

responses_df = pd.DataFrame(response_rows)
responses_df.head()

,custom_id,final_name
0,req1,Hey Maya
1,req2,Hello
2,req3,Hello
3,req4,Hello
4,req5,Hey therelentlessroyal


In [7]:
email_to_name_df = requests_df.merge(
    responses_df,
    on="custom_id",
    how="inner"
)

# dup_emails = requests_df["email"].duplicated().sum()
# assert dup_emails == 0, f"Duplicate emails detected: {dup_emails}"

email_to_name_df.head()

,custom_id,email,final_name
0,req1,mayawofsywellness@gmail.com,Hey Maya
1,req2,brevardheatad@gmail.com,Hello
2,req3,thepulsecreativelabs@gmail.com,Hello
3,req4,otherpeoplespockets@gmail.com,Hello
4,req5,therelentlessroyal@gmail.com,Hey therelentlessroyal


In [8]:
df = pd.read_csv(CSV_PATH)

df["email"] = df["email"].str.lower().str.strip()
df.head()

,email,final_name,Description,Keyword,Network,Proxy Groups,Title,URL
0,mayawofsywellness@gmail.com,NaN,I now live in New York City and I started my o...,executive coach,Instagram,RESIDENTIAL,Reintroducing Maya Wofsy Wellness: Your Holist...,https://www.instagram.com/reel/DP4DpeqkQAl/
1,brevardheatad@gmail.com,NaN,New York Flames age groups‼️Please share‼️ We ...,executive coach,Instagram,RESIDENTIAL,Introducing Firehawk Basketball's 2026 Coachin...,https://www.instagram.com/reel/DQKBrLIEV8u/
2,thepulsecreativelabs@gmail.com,NaN,"Sep 20, 2025—thepulsecreativelabs@gmail.com @t...",executive coach,Instagram,RESIDENTIAL,"The True Business of Basketball: Leadership, E...",https://www.instagram.com/reel/DO1iPR0k0Qw/-co...
3,otherpeoplespockets@gmail.com,NaN,Email me with your guest ideas at otherpeoples...,executive coach,Instagram,RESIDENTIAL,Navigating Fear with Curiosity: Empowering Pro...,https://www.instagram.com/reel/DQZ7MxzD5gR/
4,therelentlessroyal@gmail.com,NaN,Email: therelentlessroyal@gmail.com #YouthConf...,executive coach,Instagram,RESIDENTIAL,I appreciate your leadership and wisdom coach....,https://www.instagram.com/relentless_royal/p/D...


In [9]:
df_merged = df.merge(
    email_to_name_df[["email", "final_name"]],
    on="email",
    how="left",
    suffixes=("", "_llm")
)

In [10]:
df_merged["final_name"] = df_merged["final_name_llm"].combine_first(
    df_merged.get("final_name")
)

df_merged.drop(columns=["final_name_llm"], inplace=True)

In [11]:
total = len(df_merged)
filled = df_merged["final_name"].notna().sum()

print(f"Total rows: {total}")
print(f"Final_name filled: {filled}")
print(f"Missing final_name: {total - filled}")

Total rows: 3584
Final_name filled: 3569
Missing final_name: 15


In [ ]:
df_merged[df_merged["final_name"].isna()][
    ["email"]
].head(10)

KeyError: "['fullName', 'username', 'title'] not in index"

In [ ]:
df_merged.to_csv(OUTPUT_CSV_PATH, index=False)
print("✅ CSV updated successfully")